In [11]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import Conv2d, ConvTranspose2d, BatchNorm2d, BatchNorm1d, Linear, ReLU, LeakyReLU, Tanh

# 以下内容为AI（Copilot）根据GAN_Test.ipynb，以及关于该文件的问题生成。

# Generator 架构说明

## 核心目标
从 100维隐向量 生成 28×28 图像

## 完整流程

```
输入: z ∈ ℝ^100 (标准正态分布)
  ↓
[1-2] Linear(100→6272) + Reshape → (B, 128, 7, 7)
  ↓
[3-6] BN→ReLU→Upsample(×2)→Conv2d → (B, 128, 14, 14)
  ↓
[7-10] BN→ReLU→Upsample(×2)→Conv2d → (B, 64, 28, 28)
  ↓
[11-14] BN→ReLU→Conv2d→Tanh → (B, 1, 28, 28) ∈ [-1,1]
```

## 关键设计

**1. Upsample + Conv2d (不用 ConvTranspose2d)**
- ✅ 避免棋盘效应
- ✅ 参数更少 (849,793 vs 3,572,864)
- ✅ 训练更稳定

**2. Tanh 输出**
- 输出范围 [-1, 1]，匹配数据归一化
- 梯度比 Sigmoid 更好

**3. 渐进式上采样**
- 7×7 → 14×14 → 28×28
- 逐步细化，类似画草图再添细节

## 参数统计

| 层 | 操作 | 形状变化 | 参数量 |
|----|------|---------|-------|
| 1 | Linear | (100) → (6272) | 633,472 |
| 2 | Reshape | (6272) → (128,7,7) | 0 |
| 3-6 | 上采样块1 | (128,7,7) → (128,14,14) | 147,840 |
| 7-10 | 上采样块2 | (128,14,14) → (64,28,28) | 74,048 |
| 11-14 | 输出层 | (64,28,28) → (1,28,28) | 705 |

**总参数: 849,793**

In [12]:
# Generator 层 1-2: 隐向量 → 初始特征图

print("\n" + "="*80)
print("Generator 层 1-2: Linear + Reshape")
print("="*80)

batch_size = 2
z = torch.randn(batch_size, 100)

# Layer 1: Linear
fc_layer = Linear(100, 128*7*7)
out_fc = fc_layer(z)
print(f"\n[层1] Linear(100 → 6272)")
print(f"  输入: {z.shape}")
print(f"  输出: {out_fc.shape}")
print(f"  参数: {sum(p.numel() for p in fc_layer.parameters()):,}")

# Layer 2: Reshape
out_reshape = out_fc.view(batch_size, 128, 7, 7)
print(f"\n[层2] Reshape")
print(f"  输入: {out_fc.shape}")
print(f"  输出: {out_reshape.shape}  (128个 7×7 特征图)")
print(f"  这是生成过程的'种子'，后续通过上采样放大到 28×28")


Generator 层 1-2: Linear + Reshape

[层1] Linear(100 → 6272)
  输入: torch.Size([2, 100])
  输出: torch.Size([2, 6272])
  参数: 633,472

[层2] Reshape
  输入: torch.Size([2, 6272])
  输出: torch.Size([2, 128, 7, 7])  (128个 7×7 特征图)
  这是生成过程的'种子'，后续通过上采样放大到 28×28


In [13]:
# Generator 层 3-6: 第一次上采样 (7×7 → 14×14)

print("\n" + "="*80)
print("Generator 层 3-6: 上采样块 1")
print("="*80)

# Layer 3: BatchNorm2d
batch_norm_1 = BatchNorm2d(128)
out_bn_1 = batch_norm_1(out_reshape)
print(f"\n[层3] BatchNorm2d(128)")
print(f"  归一化 128 个通道")

# Layer 4: ReLU
out_relu_1 = torch.relu(out_bn_1)
print(f"\n[层4] ReLU")
print(f"  非线性激活: f(x) = max(0, x)")

# Layer 5: Upsample
upsample_1 = nn.Upsample(scale_factor=2, mode='nearest')
out_up_1 = upsample_1(out_relu_1)
print(f"\n[层5] Upsample(scale_factor=2)")
print(f"  {out_relu_1.shape} → {out_up_1.shape}")
print(f"  最近邻插值: 每个像素复制成 2×2 块")
print(f"  参数量: 0 (无需训练)")

# Layer 6: Conv2d
conv_1 = Conv2d(128, 128, 3, padding=1)
out_conv_1 = conv_1(out_up_1)
print(f"\n[层6] Conv2d(128→128, 3×3)")
print(f"  {out_up_1.shape} → {out_conv_1.shape}")
print(f"  平滑上采样的锯齿效果")
print(f"  参数: {sum(p.numel() for p in conv_1.parameters()):,}")


Generator 层 3-6: 上采样块 1

[层3] BatchNorm2d(128)
  归一化 128 个通道

[层4] ReLU
  非线性激活: f(x) = max(0, x)

[层5] Upsample(scale_factor=2)
  torch.Size([2, 128, 7, 7]) → torch.Size([2, 128, 14, 14])
  最近邻插值: 每个像素复制成 2×2 块
  参数量: 0 (无需训练)

[层6] Conv2d(128→128, 3×3)
  torch.Size([2, 128, 14, 14]) → torch.Size([2, 128, 14, 14])
  平滑上采样的锯齿效果
  参数: 147,584


In [14]:
# Generator 层 7-10: 第二次上采样 (14×14 → 28×28)

print("\n" + "="*80)
print("Generator 层 7-10: 上采样块 2")
print("="*80)

batch_norm_2 = BatchNorm2d(128)
out_bn_2 = batch_norm_2(out_conv_1)
out_relu_2 = torch.relu(out_bn_2)
print(f"\n[层7-8] BatchNorm2d + ReLU")

upsample_2 = nn.Upsample(scale_factor=2, mode='nearest')
out_up_2 = upsample_2(out_relu_2)
print(f"\n[层9] Upsample(×2)")
print(f"  {out_relu_2.shape} → {out_up_2.shape}")
print(f"  已达到 MNIST 目标尺寸 28×28")

conv_2 = Conv2d(128, 64, 3, padding=1)
out_conv_2 = conv_2(out_up_2)
print(f"\n[层10] Conv2d(128→64, 3×3)")
print(f"  {out_up_2.shape} → {out_conv_2.shape}")
print(f"  通道减半，为输出做准备")
print(f"  参数: {sum(p.numel() for p in conv_2.parameters()):,}")


Generator 层 7-10: 上采样块 2

[层7-8] BatchNorm2d + ReLU

[层9] Upsample(×2)
  torch.Size([2, 128, 14, 14]) → torch.Size([2, 128, 28, 28])
  已达到 MNIST 目标尺寸 28×28

[层10] Conv2d(128→64, 3×3)
  torch.Size([2, 128, 28, 28]) → torch.Size([2, 64, 28, 28])
  通道减半，为输出做准备
  参数: 73,792


In [15]:
# Generator 层 11-14: 输出层

print("\n" + "="*80)
print("Generator 层 11-14: 生成最终图像")
print("="*80)

batch_norm_3 = BatchNorm2d(64)
out_bn_3 = batch_norm_3(out_conv_2)
out_relu_3 = torch.relu(out_bn_3)
print(f"\n[层11-12] BatchNorm2d + ReLU")

conv_final = Conv2d(64, 1, 3, padding=1)
out_conv_final = conv_final(out_relu_3)
print(f"\n[层13] Conv2d(64→1, 3×3)")
print(f"  {out_relu_3.shape} → {out_conv_final.shape}")
print(f"  生成单通道灰度图")
print(f"  参数: {sum(p.numel() for p in conv_final.parameters()):,}")

tanh = Tanh()
out_tanh = tanh(out_conv_final)
print(f"\n[层14] Tanh")
print(f"  输出范围: [{out_tanh.min():.4f}, {out_tanh.max():.4f}] ∈ [-1, 1]")
print(f"\n✓ Generator 完成! 从 100维隐向量 生成 28×28 图像")


Generator 层 11-14: 生成最终图像

[层11-12] BatchNorm2d + ReLU

[层13] Conv2d(64→1, 3×3)
  torch.Size([2, 64, 28, 28]) → torch.Size([2, 1, 28, 28])
  生成单通道灰度图
  参数: 577

[层14] Tanh
  输出范围: [-0.8869, 0.7948] ∈ [-1, 1]

✓ Generator 完成! 从 100维隐向量 生成 28×28 图像


# Discriminator 架构说明

## 核心目标
判别图像真假，输出概率 [0, 1]

## 完整流程

```
输入: 图像 (1, 28, 28) ∈ [-1, 1]
  ↓
[1-3] Conv2d(stride=2)→LeakyReLU→Dropout → (64, 14, 14)
  ↓
[4-7] Conv2d→BN→LeakyReLU→Dropout → (128, 7, 7)
  ↓
[8-11] Conv2d→BN→LeakyReLU→Dropout → (256, 3, 3)
  ↓
[12-14] Conv2d→BN→LeakyReLU → (512, 1, 1)
  ↓
[15-17] Flatten→Linear→Sigmoid → 概率 ∈ [0, 1]
```

## 关键设计

**1. 第一层不用 BatchNorm**
- 保留原始输入特征，便于判别

**2. LeakyReLU(0.2)**
- 负值允许通过（×0.2）
- 避免"神经元死亡"，梯度流动更好

**3. Dropout2d(0.3)**
- 随机丢弃 30% 的特征图通道
- 防止过拟合

**4. Sigmoid 输出**
- 0.0 = 假图像
- 1.0 = 真图像

## 参数统计

| 层 | 操作 | 形状变化 | 参数量 |
|----|------|---------|-------|
| 1-3 | 初始特征 | (1,28,28) → (64,14,14) | 1,088 |
| 4-7 | 卷积块1 | (64,14,14) → (128,7,7) | 131,456 |
| 8-11 | 卷积块2 | (128,7,7) → (256,3,3) | 525,056 |
| 12-14 | 卷积块3 | (256,3,3) → (512,1,1) | 2,098,688 |
| 15-17 | 分类层 | (512,1,1) → (1,) | 513 |

**总参数: 2,755,801**

In [16]:
# Discriminator 层 1-3: 初始特征提取

print("\n" + "="*80)
print("Discriminator 层 1-3: 初始特征提取")
print("="*80)

batch_size = 2
input_img = torch.randn(batch_size, 1, 28, 28)

# Layer 1: Conv2d (第一层不用BN)
conv_d_1 = Conv2d(1, 64, 4, 2, 1)
out_conv_d_1 = conv_d_1(input_img)
print(f"\n[层1] Conv2d(1→64, 4×4, stride=2)")
print(f"  {input_img.shape} → {out_conv_d_1.shape}")
print(f"  28×28 → 14×14 下采样")
print(f"  参数: {sum(p.numel() for p in conv_d_1.parameters()):,}")
print(f"  ⚠️ 第一层不用 BatchNorm，保留原始特征")

# Layer 2: LeakyReLU
leaky_relu = LeakyReLU(0.2)
out_lrelu_d_1 = leaky_relu(out_conv_d_1)
print(f"\n[层2] LeakyReLU(0.2)")
print(f"  f(x) = x if x>0 else 0.2×x")
print(f"  允许负梯度，防止神经元死亡")

# Layer 3: Dropout2d
dropout_d_1 = nn.Dropout2d(0.3)
dropout_d_1.train()
out_dropout_d_1 = dropout_d_1(out_lrelu_d_1)
print(f"\n[层3] Dropout2d(0.3)")
print(f"  随机丢弃 30% 的特征图通道")
print(f"  防止过拟合")


Discriminator 层 1-3: 初始特征提取

[层1] Conv2d(1→64, 4×4, stride=2)
  torch.Size([2, 1, 28, 28]) → torch.Size([2, 64, 14, 14])
  28×28 → 14×14 下采样
  参数: 1,088
  ⚠️ 第一层不用 BatchNorm，保留原始特征

[层2] LeakyReLU(0.2)
  f(x) = x if x>0 else 0.2×x
  允许负梯度，防止神经元死亡

[层3] Dropout2d(0.3)
  随机丢弃 30% 的特征图通道
  防止过拟合


In [17]:
# Discriminator 层 4-7: 卷积块 1 (14×14 → 7×7)

print("\n" + "="*80)
print("Discriminator 层 4-7: 卷积块 1")
print("="*80)

conv_d_2 = Conv2d(64, 128, 4, 2, 1)
out_conv_d_2 = conv_d_2(out_dropout_d_1)
print(f"\n[层4] Conv2d(64→128, 4×4, stride=2)")
print(f"  {out_dropout_d_1.shape} → {out_conv_d_2.shape}")
print(f"  参数: {sum(p.numel() for p in conv_d_2.parameters()):,}")

batch_norm_d_1 = BatchNorm2d(128)
out_bn_d_1 = batch_norm_d_1(out_conv_d_2)
out_lrelu_d_2 = nn.functional.leaky_relu(out_bn_d_1, 0.2)
print(f"\n[层5-6] BatchNorm2d + LeakyReLU")

dropout_d_2 = nn.Dropout2d(0.3)
dropout_d_2.train()
out_dropout_d_2 = dropout_d_2(out_lrelu_d_2)
print(f"\n[层7] Dropout2d(0.3)")


Discriminator 层 4-7: 卷积块 1

[层4] Conv2d(64→128, 4×4, stride=2)
  torch.Size([2, 64, 14, 14]) → torch.Size([2, 128, 7, 7])
  参数: 131,200

[层5-6] BatchNorm2d + LeakyReLU

[层7] Dropout2d(0.3)


In [18]:
# Discriminator 层 8-11: 卷积块 2 (7×7 → 3×3)

print("\n" + "="*80)
print("Discriminator 层 8-11: 卷积块 2")
print("="*80)

conv_d_3 = Conv2d(128, 256, 4, 2, 1)
out_conv_d_3 = conv_d_3(out_dropout_d_2)
print(f"\n[层8] Conv2d(128→256, 4×4, stride=2)")
print(f"  {out_dropout_d_2.shape} → {out_conv_d_3.shape}")
print(f"  参数: {sum(p.numel() for p in conv_d_3.parameters()):,}")

batch_norm_d_2 = BatchNorm2d(256)
out_bn_d_2 = batch_norm_d_2(out_conv_d_3)
out_lrelu_d_3 = nn.functional.leaky_relu(out_bn_d_2, 0.2)
print(f"\n[层9-10] BatchNorm2d + LeakyReLU")

dropout_d_3 = nn.Dropout2d(0.3)
dropout_d_3.train()
out_dropout_d_3 = dropout_d_3(out_lrelu_d_3)
print(f"\n[层11] Dropout2d(0.3)")


Discriminator 层 8-11: 卷积块 2

[层8] Conv2d(128→256, 4×4, stride=2)
  torch.Size([2, 128, 7, 7]) → torch.Size([2, 256, 3, 3])
  参数: 524,544

[层9-10] BatchNorm2d + LeakyReLU

[层11] Dropout2d(0.3)


In [19]:
# Discriminator 层 12-17: 最终分类

print("\n" + "="*80)
print("Discriminator 层 12-17: 分类输出")
print("="*80)

conv_d_4 = Conv2d(256, 512, 4, 2, 1)
out_conv_d_4 = conv_d_4(out_dropout_d_3)
print(f"\n[层12] Conv2d(256→512, 4×4, stride=2)")
print(f"  {out_dropout_d_3.shape} → {out_conv_d_4.shape}")
print(f"  压缩到 1×1 空间尺寸")
print(f"  参数: {sum(p.numel() for p in conv_d_4.parameters()):,}")

batch_norm_d_3 = BatchNorm2d(512)
out_bn_d_3 = batch_norm_d_3(out_conv_d_4)
out_lrelu_d_4 = nn.functional.leaky_relu(out_bn_d_3, 0.2)
print(f"\n[层13-14] BatchNorm2d + LeakyReLU")

out_flatten = out_lrelu_d_4.view(out_lrelu_d_4.size(0), -1)
print(f"\n[层15] Flatten")
print(f"  {out_lrelu_d_4.shape} → {out_flatten.shape}")

fc_d = Linear(512, 1)
out_fc_d = fc_d(out_flatten)
print(f"\n[层16] Linear(512→1)")
print(f"  {out_flatten.shape} → {out_fc_d.shape}")
print(f"  参数: {sum(p.numel() for p in fc_d.parameters()):,}")

sigmoid = nn.Sigmoid()
out_sigmoid = sigmoid(out_fc_d)
print(f"\n[层17] Sigmoid")
print(f"  将 logits 转换为概率 [0, 1]")
print(f"  输出: {out_sigmoid.detach().numpy().flatten()}")
print(f"\n✓ Discriminator 完成! 输出真假概率")


Discriminator 层 12-17: 分类输出

[层12] Conv2d(256→512, 4×4, stride=2)
  torch.Size([2, 256, 3, 3]) → torch.Size([2, 512, 1, 1])
  压缩到 1×1 空间尺寸
  参数: 2,097,664

[层13-14] BatchNorm2d + LeakyReLU

[层15] Flatten
  torch.Size([2, 512, 1, 1]) → torch.Size([2, 512])

[层16] Linear(512→1)
  torch.Size([2, 512]) → torch.Size([2, 1])
  参数: 513

[层17] Sigmoid
  将 logits 转换为概率 [0, 1]
  输出: [0.46009165 0.6037632 ]

✓ Discriminator 完成! 输出真假概率


---
# 第七部分：总结表格

In [21]:
# Generator vs Discriminator 对比

import pandas as pd

print("\n" + "="*100)
print("Generator vs Discriminator 架构对比")
print("="*100)

comparison_data = {
    '特征': [
        '功能',
        '输入',
        '输出',
        '主要操作',
        '空间维度',
        '激活函数',
        'BatchNorm',
        'Dropout',

    ],
    'Generator': [
        '生成图像',
        '隐向量 (100,)',
        '图像 (1,28,28) ∈[-1,1]',
        'Upsample + Conv2d',
        '7×7 → 14×14 → 28×28',
        'ReLU + Tanh',
        '每层卷积后',
        '无',

    ],
    'Discriminator': [
        '判别真假',
        '图像 (1,28,28) ∈[-1,1]',
        '概率 (1,) ∈[0,1]',
        'Conv2d (stride=2)',
        '28×28 → 14×14 → 7×7 → 3×3 → 1×1',
        'LeakyReLU + Sigmoid',
        '第一层无，其他有',
        '有 (0.3)',

    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

print("\n" + "="*100)


print("\n关键特点:")
print("  ✓ Generator 使用 Upsample+Conv (避免棋盘效应)")
print("  ✓ Discriminator 参数更多 (需要更强判别能力)")
print("  ✓ 通过对抗训练达到平衡")

df


Generator vs Discriminator 架构对比
       特征            Generator                   Discriminator
       功能                 生成图像                            判别真假
       输入           隐向量 (100,)            图像 (1,28,28) ∈[-1,1]
       输出 图像 (1,28,28) ∈[-1,1]                  概率 (1,) ∈[0,1]
     主要操作    Upsample + Conv2d               Conv2d (stride=2)
     空间维度  7×7 → 14×14 → 28×28 28×28 → 14×14 → 7×7 → 3×3 → 1×1
     激活函数          ReLU + Tanh             LeakyReLU + Sigmoid
BatchNorm                每层卷积后                        第一层无，其他有
  Dropout                    无                         有 (0.3)


关键特点:
  ✓ Generator 使用 Upsample+Conv (避免棋盘效应)
  ✓ Discriminator 参数更多 (需要更强判别能力)
  ✓ 通过对抗训练达到平衡


,特征,Generator,Discriminator
0,功能,生成图像,判别真假
1,输入,"隐向量 (100,)","图像 (1,28,28) ∈[-1,1]"
2,输出,"图像 (1,28,28) ∈[-1,1]","概率 (1,) ∈[0,1]"
3,主要操作,Upsample + Conv2d,Conv2d (stride=2)
4,空间维度,7×7 → 14×14 → 28×28,28×28 → 14×14 → 7×7 → 3×3 → 1×1
5,激活函数,ReLU + Tanh,LeakyReLU + Sigmoid
6,BatchNorm,每层卷积后,第一层无，其他有
7,Dropout,无,有 (0.3)
